In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
%matplotlib inline 

In [83]:
from sklearn.preprocessing import StandardScaler,PolynomialFeatures,OrdinalEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn import tree
from sklearn.linear_model import LinearRegression,LogisticRegression,ElasticNetCV,RidgeCV,LassoCV
from sklearn.svm import SVR

In [40]:
df=pd.read_csv('../sources/car_price_prediction_with_missing.csv')

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car ID        2250 non-null   float64
 1   Brand         2250 non-null   object 
 2   Year          2250 non-null   float64
 3   Engine Size   2250 non-null   float64
 4   Fuel Type     2250 non-null   object 
 5   Transmission  2250 non-null   object 
 6   Mileage       2250 non-null   float64
 7   Condition     2250 non-null   object 
 8   Price         2250 non-null   float64
 9   Model         2250 non-null   object 
dtypes: float64(5), object(5)
memory usage: 195.4+ KB


In [42]:
df.isnull().sum()

Car ID          250
Brand           250
Year            250
Engine Size     250
Fuel Type       250
Transmission    250
Mileage         250
Condition       250
Price           250
Model           250
dtype: int64

In [43]:
df.drop('Car ID',axis=1,inplace=True)

In [44]:
df.head(5)

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,Tesla,2016.0,2.3,Petrol,Manual,114832.0,New,26613.92,Model X
1,BMW,2018.0,4.4,Electric,Manual,143190.0,Used,14679.61,5 Series
2,Audi,2013.0,4.5,Electric,Manual,181601.0,New,44402.61,A4
3,Tesla,2011.0,4.1,Diesel,Automatic,68682.0,New,86374.33,Model Y
4,Ford,2009.0,2.6,Diesel,Manual,223009.0,Like New,73577.10,Mustang


In [45]:
columns_to_check=df.columns.tolist()

In [46]:
for column in columns_to_check: 
    print(column)
    print(df[column].unique())
    print(df[column].value_counts())

Brand
['Tesla' 'BMW' 'Audi' 'Ford' 'Honda' 'Mercedes' 'Toyota' nan]
Brand
Toyota      346
Mercedes    331
BMW         326
Audi        323
Tesla       314
Ford        307
Honda       303
Name: count, dtype: int64
Year
[2016. 2018. 2013. 2011. 2009. 2019. 2020. 2017. 2023. 2010. 2001. 2006.
 2014. 2022. 2005. 2012. 2015. 2007. 2000. 2004. 2021.   nan 2003. 2008.
 2002.]
Year
2020.0    108
2016.0    103
2022.0    103
2001.0    103
2012.0    102
2007.0    101
2003.0    100
2008.0     99
2014.0     99
2002.0     97
2021.0     96
2011.0     96
2005.0     95
2019.0     94
2018.0     92
2017.0     92
2023.0     90
2004.0     89
2000.0     86
2009.0     85
2013.0     83
2010.0     83
2006.0     77
2015.0     77
Name: count, dtype: int64
Engine Size
[2.3 4.4 4.5 4.1 2.6 2.4 4.  5.3 5.7 1.5 1.8 4.7 5.4 2.  3.9 3.  1.1 3.3
 5.8 5.2 1.9 5.9 1.  3.2 5.6 4.6 4.2 nan 2.2 2.1 1.3 1.4 5.  5.1 3.7 3.6
 2.9 2.8 5.5 4.3 4.9 3.1 2.5 3.5 4.8 1.6 3.8 2.7 1.7 3.4 1.2 6. ]
Engine Size
3.9    60
1.8    59
1.3   

In [22]:
df[df['Brand'].isnull()==True]

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2442,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2480,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
rows_to_drop=df[df['Brand'].isnull()==True].index

In [48]:
rows=rows_to_drop.tolist()

In [49]:
for row in rows: 
    df=df.drop(row,axis=0)

In [50]:
df.isnull().sum()

Brand           0
Year            0
Engine Size     0
Fuel Type       0
Transmission    0
Mileage         0
Condition       0
Price           0
Model           0
dtype: int64

In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2250 entries, 0 to 2499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         2250 non-null   object 
 1   Year          2250 non-null   float64
 2   Engine Size   2250 non-null   float64
 3   Fuel Type     2250 non-null   object 
 4   Transmission  2250 non-null   object 
 5   Mileage       2250 non-null   float64
 6   Condition     2250 non-null   object 
 7   Price         2250 non-null   float64
 8   Model         2250 non-null   object 
dtypes: float64(4), object(5)
memory usage: 175.8+ KB


In [53]:
df.describe()

,Year,Engine Size,Mileage,Price
count,2250.000000,2250.000000,2250.000000,2250.000000
mean,2011.577778,3.485467,150236.178222,52506.874391
std,6.980468,1.427690,88150.746556,27232.417079
min,2000.000000,1.000000,15.000000,5011.270000
25%,2005.000000,2.300000,71178.500000,28985.052500
50%,2012.000000,3.450000,149762.000000,53485.240000
75%,2018.000000,4.700000,226299.000000,75560.340000
max,2023.000000,6.000000,299967.000000,99982.590000


In [54]:
df.head(50)

,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,Tesla,2016.0,2.3,Petrol,Manual,114832.0,New,26613.92,Model X
1,BMW,2018.0,4.4,Electric,Manual,143190.0,Used,14679.61,5 Series
2,Audi,2013.0,4.5,Electric,Manual,181601.0,New,44402.61,A4
3,Tesla,2011.0,4.1,Diesel,Automatic,68682.0,New,86374.33,Model Y
4,Ford,2009.0,2.6,Diesel,Manual,223009.0,Like New,73577.10,Mustang
5,Audi,2019.0,2.4,Diesel,Automatic,246553.0,Like New,88969.76,Q7
6,Audi,2020.0,4.0,Electric,Automatic,135486.0,Used,63498.75,Q5
7,Tesla,2017.0,5.3,Hybrid,Automatic,83030.0,New,17381.19,Model Y
8,Honda,2023.0,5.7,Electric,Manual,120360.0,Like New,15905.62,Civic
9,Ford,2010.0,1.5,Electric,Automatic,135009.0,Like New,9560.22,Explorer


In [55]:
df['Brand'].unique()

array(['Tesla', 'BMW', 'Audi', 'Ford', 'Honda', 'Mercedes', 'Toyota'],
      dtype=object)

In [56]:
df['Model'].unique()

array(['Model X', '5 Series', 'A4', 'Model Y', 'Mustang', 'Q7', 'Q5',
       'Civic', 'Explorer', 'Model 3', 'Fiesta', 'X3', 'GLA', 'A3', 'X5',
       'C-Class', 'E-Class', 'CR-V', 'Camry', 'Accord', 'GLC', 'Corolla',
       'Fit', 'Model S', 'Prius', '3 Series', 'RAV4', 'Focus'],
      dtype=object)

In [104]:
df.groupby(['Brand','Model'])['Price'].mean().sort_values(ascending=False)

Brand     Model   
Mercedes  GLC         59907.302432
BMW       3 Series    58659.311744
          5 Series    56474.966506
Honda     Fit         55609.483671
Toyota    Camry       55281.050897
Ford      Focus       55104.703429
Tesla     Model 3     53883.158026
Toyota    Corolla     53856.738454
Tesla     Model Y     53667.360361
Honda     Accord      53615.951899
Mercedes  C-Class     53385.932414
Audi      Q5          53263.977910
Ford      Explorer    53138.291711
Audi      A3          52130.691609
Tesla     Model S     52016.052899
Audi      Q7          51979.788272
          A4          51864.724318
Ford      Fiesta      51449.422500
Tesla     Model X     51188.934535
BMW       X5          51143.619324
Toyota    RAV4        50563.257711
Mercedes  E-Class     50489.457952
BMW       X3          49845.898675
Honda     Civic       49702.850758
Mercedes  GLA         49438.343793
Toyota    Prius       48173.595227
Honda     CR-V        47974.055443
Ford      Mustang     46617.238219
N

In [64]:
df.groupby('Brand')['Model'].unique()

Brand
Audi                            [A4, Q7, Q5, A3]
BMW                 [5 Series, X3, X5, 3 Series]
Ford          [Mustang, Explorer, Fiesta, Focus]
Honda                 [Civic, CR-V, Accord, Fit]
Mercedes            [GLA, C-Class, E-Class, GLC]
Tesla       [Model X, Model Y, Model 3, Model S]
Toyota             [Camry, Corolla, Prius, RAV4]
Name: Model, dtype: object

In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2250 entries, 0 to 2499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         2250 non-null   object 
 1   Year          2250 non-null   float64
 2   Engine Size   2250 non-null   float64
 3   Fuel Type     2250 non-null   object 
 4   Transmission  2250 non-null   object 
 5   Mileage       2250 non-null   float64
 6   Condition     2250 non-null   object 
 7   Price         2250 non-null   float64
 8   Model         2250 non-null   object 
dtypes: float64(4), object(5)
memory usage: 175.8+ KB


In [66]:
df['Brand'].unique().tolist()

['Tesla', 'BMW', 'Audi', 'Ford', 'Honda', 'Mercedes', 'Toyota']

In [110]:
X=df.drop('Price',axis=1)
y=df['Price']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=15)

In [111]:
X_train['Model+Brand']=X_train['Brand'] + "_" + X_train['Model']

In [112]:
X_test['Model+Brand']=X_test['Brand'] + "_" + X_test['Model']

In [116]:
X_test.drop(['Brand','Model'],axis=1,inplace=True)
X_train.drop(['Brand','Model'],axis=1,inplace=True)

In [117]:
X_train

,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Model+Brand
1658,2004.0,5.6,Hybrid,Automatic,166052.0,Used,Honda_Civic
2456,2009.0,3.6,Electric,Manual,118194.0,Like New,BMW_5 Series
2142,2011.0,5.7,Hybrid,Automatic,30924.0,Used,Ford_Fiesta
1640,2014.0,5.1,Petrol,Manual,160966.0,Used,Audi_Q7
2023,2003.0,1.3,Electric,Automatic,265880.0,Like New,Honda_Accord
...,...,...,...,...,...,...,...
1045,2006.0,1.4,Hybrid,Automatic,108138.0,New,Honda_Accord
698,2004.0,4.0,Diesel,Automatic,116251.0,Like New,BMW_X3
2373,2018.0,2.1,Hybrid,Manual,214209.0,Used,Toyota_Corolla
1937,2014.0,1.4,Electric,Automatic,158964.0,Like New,Tesla_Model S


In [121]:
ordered_categories = X_train.assign(Price=y_train).groupby('Model+Brand')['Price'].mean().sort_values().index.tolist()

In [122]:
categorial_columns=['Fuel Type','Transmission','Condition','Model+Brand']
numeric_columns=['Year','Engine Size','Mileage','Price']
ordinal_encoder=OrdinalEncoder(categories=[
                                           df['Fuel Type'].unique().tolist(),
                                           df['Transmission'].unique().tolist(),
                                           df['Condition'].unique().tolist(),
                                            ordered_categories
                                          ])
preprocessor=ColumnTransformer(transformers=[('deneme',ordinal_encoder,categorial_columns)],remainder='passthrough')

In [123]:
X_train_transformed=preprocessor.fit_transform(X_train)
X_test_transformed=preprocessor.transform(X_test)

In [124]:
pd.DataFrame(X_train_transformed)

,0,1,2,3,4,5,6
0,3.0,1.0,1.0,8.0,2004.0,5.6,166052.0
1,1.0,0.0,2.0,25.0,2009.0,3.6,118194.0
2,3.0,1.0,1.0,7.0,2011.0,5.7,30924.0
3,0.0,0.0,1.0,15.0,2014.0,5.1,160966.0
4,1.0,1.0,2.0,22.0,2003.0,1.3,265880.0
...,...,...,...,...,...,...,...
1795,3.0,1.0,0.0,22.0,2006.0,1.4,108138.0
1796,2.0,1.0,2.0,6.0,2004.0,4.0,116251.0
1797,3.0,0.0,1.0,13.0,2018.0,2.1,214209.0
1798,1.0,1.0,2.0,11.0,2014.0,1.4,158964.0


In [125]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train_transformed)
X_test_scaled=scaler.transform(X_test_transformed)

In [126]:
X_train_scaled

array([[ 1.36685866,  1.02133873,  0.00686172, ..., -1.09321636,
         1.49550741,  0.17646386],
       [-0.43427281, -0.9791071 ,  1.24197125, ..., -0.37598737,
         0.09116521, -0.36454309],
       [ 1.36685866,  1.02133873,  0.00686172, ..., -0.08909578,
         1.56572452, -1.35107985],
       ...,
       [ 1.36685866, -0.9791071 ,  0.00686172, ...,  0.9150248 ,
        -0.96209143,  0.72085083],
       [-0.43427281,  1.02133873,  1.24197125, ...,  0.34124161,
        -1.4536112 ,  0.09633813],
       [-1.33483854, -0.9791071 ,  0.00686172, ...,  1.0584706 ,
        -1.31317698,  0.57669686]])

In [127]:
y_train

1658    95620.03
2456    94713.45
2142    69588.36
1640    94355.25
2023    66780.72
          ...   
1045    34629.56
698     55381.61
2373    95242.86
1937    74090.14
2450    47152.53
Name: Price, Length: 1800, dtype: float64

In [128]:
reg_tree=DecisionTreeRegressor()
reg_tree.fit(X_train_scaled,y_train)

DecisionTreeRegressor()

In [129]:
y_pred=reg_tree.predict(X_test_scaled)

In [130]:
print("mse:",mean_squared_error(y_test,y_pred))
print("msa:",mean_absolute_error(y_test,y_pred))
print("score:",r2_score(y_test,y_pred))

mse: 1509660922.0108404
msa: 31104.011088888892
score: -0.9422430213352828


In [131]:
liner=LinearRegression()
liner.fit(X_train_scaled,y_train)
y_pred_liner=liner.predict(X_test_scaled)
print("mse:",mean_squared_error(y_test,y_pred_liner))
print("msa:",mean_absolute_error(y_test,y_pred_liner))
print("score:",r2_score(y_test,y_pred_liner))

mse: 789560547.2915102
msa: 24380.79408060102
score: -0.015803244649125636


In [134]:
#hypertuning: 
params={'criterion':['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
        'splitter':['best','random'],
        'max_depth':[5,10,20,30,40,50,100],
         'max_features':['sqrt','log2']
       }

In [135]:
grid=GridSearchCV(estimator=DecisionTreeRegressor(),param_grid=params,cv=5,scoring='r2')
grid.fit(X_train_scaled,y_train)
y_pred_grid=grid.predict(X_test_scaled)
print("mse:",mean_squared_error(y_test,y_pred_grid))
print("msa:",mean_absolute_error(y_test,y_pred_grid))
print("score:",r2_score(y_test,y_pred_grid))

mse: 812618573.3732399
msa: 24532.89574035166
score: -0.04546837646121138


NameError: name 'g' is not defined